In [1]:
import pandas as pd

In [11]:
import pandas as pd

# 1. Adım: Tüm dosyaları tek tek okuma
demo = pd.read_sas(r'C:\Users\kadir\OneDrive\Masaüstü\korelaktal\1999-2000\DEMO.xpt')
mcq = pd.read_sas(r'C:\Users\kadir\OneDrive\Masaüstü\korelaktal\1999-2000\MCQ.xpt')
alq = pd.read_sas(r'C:\Users\kadir\OneDrive\Masaüstü\korelaktal\1999-2000\ALQ.xpt')
smq = pd.read_sas(r'C:\Users\kadir\OneDrive\Masaüstü\korelaktal\1999-2000\SMQ.xpt')
whq = pd.read_sas(r'C:\Users\kadir\OneDrive\Masaüstü\korelaktal\1999-2000\WHQ.xpt')
paq = pd.read_sas(r'C:\Users\kadir\OneDrive\Masaüstü\korelaktal\1999-2000\PAQ.xpt')

# Laboratuvar verileri
lab11 = pd.read_sas(r'C:\Users\kadir\OneDrive\Masaüstü\korelaktal\1999-2000\LAB11.xpt')
lab18 = pd.read_sas(r'C:\Users\kadir\OneDrive\Masaüstü\korelaktal\1999-2000\LAB18.xpt')
lab18t4 = pd.read_sas(r'C:\Users\kadir\OneDrive\Masaüstü\korelaktal\1999-2000\LAB18T4.xpt')
lab25 = pd.read_sas(r'C:\Users\kadir\OneDrive\Masaüstü\korelaktal\1999-2000\LAB25.xpt')

# 2. Adım: Verileri SEQN (Hasta ID) üzerinden birleştirme (Merge)
# Temel demografi tablosu ile başlıyoruz
df = demo.copy()

# Diğer tüm tabloları sırasıyla ekliyoruz
dosyalar = [mcq, alq, smq, whq, paq, lab11, lab18, lab18t4, lab25]

for dosya in dosyalar:
    # how='outer' kullanarak hiçbir hasta verisini kaybetmiyoruz
    df = pd.merge(df, dosya, on='SEQN', how='outer')

# 3. Adım: Birleştirilmiş veriyi CSV olarak kaydetme (Daha sonra kolay kullanım için)
df.to_csv('NHANES_Kolorektal_Kanser_Verisi.csv', index=False)

print("Dosyalar başarıyla birleştirildi!")
print("Toplam Satır (Hasta) ve Sütun (Özellik) Sayısı:", df.shape)

Dosyalar başarıyla birleştirildi!
Toplam Satır (Hasta) ve Sütun (Özellik) Sayısı: (9965, 490)


In [13]:
print(df.columns)

Index(['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIDEXMON', 'RIAGENDR', 'RIDAGEYR',
       'RIDAGEMN', 'RIDAGEEX', 'RIDRETH1', 'RIDRETH2',
       ...
       'LBDBANO', 'LBXRBCSI', 'LBXHGB', 'LBXHCT', 'LBXMCVSI', 'LBXMCHSI',
       'LBXMC', 'LBXRDW', 'LBXPLTSI', 'LBXMPSI'],
      dtype='object', length=490)


In [ ]:
import pandas as pd
import numpy as np

# MÜKEMMEL KDS VERİ SETİ İÇİN KOLON HARİTASI
kolon_haritasi = {
    # --- 1. DEMOGRAFİK VE RİSK FAKTÖRLERİ (Girdiler) ---
    'SEQN': 'Hasta_ID',
    'RIDAGEYR': 'Yas',
    'RIAGENDR': 'Cinsiyet',          # 1: Erkek, 2: Kadın
    'BMXBMI': 'Vucut_Kitle_Indeksi', # Obezite riski için (BMX dosyasından)
    'SMQ020': 'Sigara_Kullanimi',    # Hayatında 100'den fazla sigara içti mi?
    
    # --- 2. LABORATUVAR / KAN BULGULARI (Girdiler) ---
    'LBXHGB': 'Hemoglobin',          # Tümör kanamasına bağlı "Anemi" tespiti için 1 numara
    'LBXHCT': 'Hematokrit',
    'LBXWBCSI': 'Akyuvar_WBC',       # Vücuttaki iltihap (enflamasyon) seviyesi
    
    # --- 3. KLİNİK SEMPTOMLAR / BELİRTİLER (Girdiler) ---
    'BAQ080': 'Rektal_Kanama',       # Dışkıda kan gördünüz mü? (BAQ anketinden)
    'BAQ070': 'Bagirsak_Kacirmasi',  # Bağırsak alışkanlığında değişim / ishal (BAQ anketinden)
    'WHD020': 'Kilo_Verme_Durumu',   # Son 1 yılda kilo verdiniz mi?
    'HUQ010': 'Genel_Saglik_Hissi',  # Kendinizi nasıl hissediyorsunuz? (1: Mükemmel, 5: Kötü)
    
    # --- 4. HEDEF ETİKETLER / LABELS (Çıktılar - MCQ dosyasından) ---
    'MCQ220': 'Kanser_Gecmisi',      # Hiç kanser oldunuz mu? (1: Evet, 2: Hayır)
    'MCQ230A': 'Kanser_Turu_1',      # 1. Kanser türü
    'MCQ230B': 'Kanser_Turu_2',      # 2. Kanser türü (Varsa)
    'MCQ230C': 'Kanser_Turu_3'       # 3. Kanser türü (Varsa)
}

# (Varsayalım ki df isimli birleştirilmiş tablonuz var)
# İhtiyacımız olan kolonları filtreleyip isimlerini çevirelim
secilen_kolonlar = [col for col in kolon_haritasi.keys() if col in df.columns]
df_temiz = df[secilen_kolonlar].rename(columns=kolon_haritasi)

# =====================================================================
# HEDEF ETİKETİ (LABEL) OLUŞTURMA MANTIĞI
# Yapay zekanın "Sağlıklı" ve "Kanser" ayrımını öğrenmesi için
# =====================================================================

# NHANES veri sözlüğüne göre:
# 14 kodu -> Kolon Kanseri
# 31 kodu -> Rektum Kanseri

def etiket_olustur(row):
    # Eğer kişinin kanser geçmişi yoksa (2 = Hayır) -> Sınıf 0 (Sağlıklı / Risksiz)
    if row['Kanser_Gecmisi'] == 2.0:
        return 0 
    
    # Eğer kişi kanser olmuşsa ve bu kanser Kolon (14) veya Rektum (31) ise -> Sınıf 1 (Kolorektal Kanser)
    kanser_turleri = [row['Kanser_Turu_1'], row['Kanser_Turu_2'], row['Kanser_Turu_3']]
    if 14.0 in kanser_turleri or 31.0 in kanser_turleri:
        return 1
    
    # Diğer kanser türlerine sahip olanları modelin kafasını karıştırmamak için veri setinden çıkaracağız (NaN yapıyoruz)
    return np.nan

# Hedef etiketimizi (Target/Label) oluşturalım
df_temiz['Hastalik_Durumu_LABEL'] = df_temiz.apply(etiket_olustur, axis=1)

# Kolorektal kanser harici diğer kanserleri (NaN olanları) silelim
df_temiz = df_temiz.dropna(subset=['Hastalik_Durumu_LABEL'])

# Artık Kanser Türü kolonlarına ihtiyacımız kalmadı, hedef etiketimizi ürettik, silelim.
df_temiz = df_temiz.drop(columns=['Kanser_Gecmisi', 'Kanser_Turu_1', 'Kanser_Turu_2', 'Kanser_Turu_3'])

print(df_temiz['Hastalik_Durumu_LABEL'].value_counts())
print("\nVeri seti makine öğrenmesine %100 hazır!")

ValueError: Cannot set a DataFrame without columns to the column Hastalik_Durumu_LABEL